In [1]:
%pip install "numpy" "opencv-python" -q
%pip install git+https://www.github.com/mouseland/cellpose.git
%pip install pyocclient -q

  Cloning https://www.github.com/mouseland/cellpose.git to /tmp/pip-req-build-pdgibzew
  Running command git clone --filter=blob:none --quiet https://www.github.com/mouseland/cellpose.git /tmp/pip-req-build-pdgibzew
  Resolved https://www.github.com/mouseland/cellpose.git to commit f878e3eccd3988def3e061785faccd059d71eb26
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 116.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 104.9 MB/s eta 0:00:0000:0100:01
  Created wheel for cellpose: filename=cellpose-4.1.2.dev30+gf878e3ecc-py3-none-any.whl size=215061 sha256=e564f567343f902aeaae3b058c02dde6dfbcbc9d99bb7f36fce42ec9a15889a7
  Stored in directory: /tmp/pip-ephem-wheel-cache-9m43kndo/wheels/df/b6/31/a3013c44290eabb46f4c06d1efb19744124fcad2d59684ec5e
Successfully built cellpose
  Preparing metadata (setup.py) ... done


In [ ]:
# USEFULL TO UPLOAD DATA TO OWNCLLOUD, BUT NOT NEEDED FOR THE TOOL IF DATA IS ALREADY ON THE SERVER

# import owncloud, getpass, cellpose

# # Config
# url, user = 'url', 'user'
# oc_session = owncloud.Client(url)
# oc_session.login(user, getpass.getpass(f"PW {user}: "))
# oc_session.get_file('/travail/Mines/DIMA/Segmentation/scripts/tool.py', 'tool.py')

# import tool
# oc = tool.Owncloud(oc_session)

# print("Tool chargé et prêt.")

/home/gabriel/.conda/envs/ML/lib/python3.14/site-packages/owncloud/owncloud.py:602: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
  :param \*\*kwargs: optional arguments that ``put_file`` accepts
/home/gabriel/.conda/envs/ML/lib/python3.14/site-packages/owncloud/owncloud.py:636: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
  :param \*\*kwargs: optional arguments that ``put_file`` accepts
/home/gabriel/.conda/envs/ML/lib/python3.14/site-packages/owncloud/owncloud.py:1746: SyntaxWarning: "\*" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\*"? A raw string is also an option.
  :param \*\*kwargs: optional arguments that ``requests.Request.request`` accepts
/home/gabriel/.conda/envs/ML/lib/python3.14/site-packages/owncloud/owncloud.py:1766: SyntaxW

MissingSchema: Invalid URL 'url/ocs/v1.php/cloud/capabilities': No scheme supplied. Perhaps you meant https://url/ocs/v1.php/cloud/capabilities?

In [ ]:
# oc.upload_data()

In [ ]:
# oc.upload_scripts()

In [ ]:
import cv2
import matplotlib.pyplot as plt
import string
from pathlib import Path
from cellpose import models
import tool
import os

ev = tool.Stats()

# print("Dossier courant :", os.getcwd())
# os.chdir('./Segmentation')
print("Dossier courant :", os.getcwd())
viz = tool.CellVisualizer()

# --- PARAMÈTRES ---
target = 'Cell'
current_index = '001'
mask_folder = Path('./data/mask/')

# 1. On liste tes masques finetunés
finetuned_paths = sorted(list(mask_folder.rglob(f"*{target}*")))
finetuned_paths.sort(key=lambda p: int(p.stem.split('_')[-1]))
print(f"Masques finetunés trouvés : {[p.name for p in finetuned_paths]}")
print(f"Nombre de masques finetunés : {len(finetuned_paths)}")

# Nombre de colonnes : Raw + 0-shot + Tes modèles
num_cols = 2 + len(finetuned_paths)
fig, axes = plt.subplots(1, num_cols, figsize=(num_cols * 5, 5))

# --- AXE 0 : L'IMAGE RAW ---
viz.plot(i='001', target='Image', ax=axes[0], show=False)

# # --- AXE 1 : LE 0-SHOT (LIVE) ---
print("Calcul du 0-shot en live...")
model_type = 'cyto3' if target == 'Cell' else 'nuclei'

model_0shot = models.CellposeModel(gpu=False)

# Récupération de l'image brute directement via ta méthode !
raw_img = viz.load_data(current_index, 'image') 

true_mask = viz.load_data(current_index, target = target)

# Inférence sur l'image récupérée
masks_0shot, _, _= model_0shot.eval(raw_img, diameter=None, channels=[0,0])
perf = ev.summary_perf(true_mask, masks_0shot, iou_threshold=0.5)

# Affichage avec ta méthode
viz.plot_overlay(i=current_index, mask_pred=masks_0shot, ax=axes[1], show=False, target=target, annotation=True, prediction=True)

# --- AXES SUIVANTS : TES MODÈLES FINETUNÉS ---
for idx, p in enumerate(finetuned_paths):
    ax_idx = idx + 2
    
    # Lecture du masque sauvegardé
    mask_array = cv2.imread(str(p), cv2.IMREAD_UNCHANGED)
    
    # Affichage avec ta méthode
    viz.plot_overlay(i=current_index, mask_pred=mask_array, ax=axes[ax_idx], show=False, target=target, annotation=True, prediction=True)
    

# --- FINITIONS (LETTRES A, B, C...) ---
for n, ax in enumerate(axes):
    ax.text(0.02, 0.95, string.ascii_uppercase[n], transform=ax.transAxes, 
            color='white', fontsize=20, fontweight='bold', va='top')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f"Panel_{target}_{current_index}.png", dpi=300, facecolor='black', bbox_inches='tight')
plt.show()

Dossier courant : /content
Masques finetunés trouvés : ['cellpose3_Cell_pred_3.bmp', 'cellpose4-1-channel_Cell_pred_4.bmp', 'cellpose4-2-channels-raw_Cell_pred_5.bmp', 'cellpose4-2-channels-top-hat_Cell_pred_6.bmp', 'cellpose4-2-channels-nuc-pred_Cell_pred_7.bmp']
Nombre de masques finetunés : 5
Calcul du 0-shot en live...


In [ ]:
import os, shutil
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tool

from cellpose import core, utils, io, models, metrics, train, dynamics, transforms, plot
from glob import glob

import importlib
importlib.reload(tool)
import tool

visualizer = tool.CellVisualizer()

In [6]:

oc.upload_models()

In [7]:
ids_test = [str(i).zfill(3) for i in range(1,333)]


In [ ]:

from cellpose import models, io

bench_configs = [
    {
        "label": "raw_2channel_cellpose4",
        "target": "Cell",
        "load_method": "channel",
        "mode": "raw",
        "params": {"channels": [1, 2], "flow_threshold": 0.5, "cellprob_threshold": 0.0},
        "model_path": "./models/cyto4_40_raw.pth" 
     }
    ,
    {
        "label": "Nuclei_cellpose4",
        "target": "Nuc",
        "load_method": "data",
        "params": {"channels": [0, 0], "flow_threshold": 0.4, "cellprob_threshold": -0.5},
        "model_path": "./models/cellpose4_nuclei.pth"
    }
]

# Création du dossier racine 
base_output = os.path.abspath("./results/SB_332/")
os.makedirs(base_output, exist_ok=True)

for config in bench_configs:
    print(f"\nInférence : {config['label']}...")
    
    model_root = os.path.join(base_output, config['label'])
    output_dir_masks = os.path.join(model_root, "masks")
    output_dir_viz = os.path.join(model_root, "viz")
    
    os.makedirs(output_dir_masks, exist_ok=True)
    os.makedirs(output_dir_viz, exist_ok=True)
    
    model = models.CellposeModel(gpu=True, pretrained_model=config['model_path'])
    
    if config['load_method'] == "channel":
        X_val_multi, _ = visualizer.load_channel(ids_test, mode=config['mode'], target= 'Image')

    for idx, img_id in enumerate(ids_test):
            if config['load_method'] == "channel":
                img = X_val_multi[idx]
            else:
                img = visualizer.load_data(img_id, target='Image')
                
            # Inférence Cellpose
            masks, _, _ = model.eval(img, **config['params'])
            
            # Sauvegarde du masque
            save_path_mask = os.path.join(output_dir_masks, f"{img_id}_{config['target']}_pred.bmp")
            cv2.imwrite(save_path_mask, masks.astype(np.uint16)) 
            
            # Grille de visualisation individuelle (1 ligne, 3 colonnes)
            fig, axes = plt.subplots(1, 3, figsize=(24, 8))
            ax1, ax2, ax3 = axes.flatten()
            
            # Affichage Image Entrée
            visualizer.plot(i=img_id, target='Image', ax=ax1, show=False)
            ax1.set_title("A. Image Entrée")

            visualizer.plot_overlay(i=img_id, mask_pred=masks, ax=ax2, show=False, annotation=False, prediction=True)
            ax2.set_title(f"B. Prédiction Cellpose - {config['target']})")

            visualizer.plot(data=masks, target= f"{config['target']}_pred", ax=ax3, show=False)
            ax3.set_title(f"C. masques predits - {config['target']}")

            titre_rapport = f"Résultat Inférence : {img_id} ({config['label']})"
            fig.suptitle(titre_rapport, fontsize=16, fontweight='bold', y=0.98)
            plt.tight_layout()
            fig.subplots_adjust(top=0.90)
            
            save_path_viz = os.path.join(output_dir_viz, f"{img_id}_{config['target']}_viz.png")
            plt.savefig(save_path_viz, dpi=150, bbox_inches='tight')
            plt.close(fig)

# --- EXPORT GLOBAL ---
print("\nExportation du dossier d'inférence...")
try:
    chemin_export_global = "/travail/Mines/DIMA/Segmentation/data/prediction/332-SG"
    oc.download_path(base_output, chemin_export_global)
    print(f"Export réussi vers : {chemin_export_global}")
except Exception as e:
    print(f"Erreur lors de l'export global : {e}")

print(f"\nProcessus terminé. Les résultats sont sauvegardés dans : {base_output}")


Inférence : raw_2channel_cellpose4...

Inférence : Nuclei_cellpose4...

Exportation du dossier d'inférence...
Erreur lors de l'export global : HTTP error: 504

Processus terminé. Les résultats sont sauvegardés dans : /content/results/SB_332


In [10]:
chemin_export_global = "/travail/Mines/DIMA/Segmentation/data/prediction/332-SG"
oc.download_path(base_output, chemin_export_global)
print(f"Export réussi vers : {chemin_export_global}")

HTTPResponseError: HTTP error: 504